# RénoSim — Démo Phase 2 : gestes de rénovation & économie

Maison étiquette **G** (fioul, non isolée, années 1960-70, 100 m², zone H1a) →
scénarios de rénovation avec **coût (fourchette), économies annuelles, ΔCO₂ et
temps de retour simple**.

In [1]:
from renosim import (
    HeatingGeneratorType, HeatingReplacement, RoofInsulation, WallInsulation,
    WindowReplacement, DHWUpgrade, VentilationUpgrade,
    assess_scenario, predefined_bundles, simulate,
)
from tests.test_reference_cases import house_1970_oil_uninsulated

house = house_1970_oil_uninsulated()
before = simulate(house)
print(f"Situation actuelle : étiquette {before.label}, "
      f"{round(before.primary_energy_kwh_m2)} kWhep/m²/an, "
      f"{round(before.annual_cost_eur)} €/an, "
      f"{round(before.co2_kg_m2)} kgCO₂/m²/an")

Situation actuelle : étiquette G, 639 kWhep/m²/an, 5864 €/an, 205 kgCO₂/m²/an


## Gestes individuels

In [2]:
gestes = {
    "Isolation combles (R=7)": [RoofInsulation()],
    "Isolation murs (R=3,7)": [WallInsulation()],
    "Fenêtres double VIR": [WindowReplacement()],
    "PAC air-eau": [HeatingReplacement(new_generator=HeatingGeneratorType.HEAT_PUMP_AIR_WATER)],
    "Chauffe-eau thermodyn.": [DHWUpgrade()],
    "VMC hygro B": [VentilationUpgrade()],
}
print(f"{'Geste':26s} {'Étiq.':6s} {'Coût (€)':>18s} {'Écon. €/an':>11s} {'Retour (ans)':>14s}")
for nom, mesures in gestes.items():
    a = assess_scenario(house, mesures)
    cout = f"{round(a.investment_low_eur):,}-{round(a.investment_high_eur):,}".replace(",", " ")
    retour = (f"{a.payback_years_low:.0f}-{a.payback_years_high:.0f}"
              if a.payback_years_high != float("inf") else "> 30")
    print(f"{nom:26s} {a.after.label:6s} {cout:>18s} {round(a.annual_savings_eur):>11,} {retour:>14s}"
          .replace(",", " "))

Geste                      Étiq.            Coût (€)  Écon. €/an   Retour (ans)
Isolation combles (R=7)    G             1 700-9 300       1 198            1-8
Isolation murs (R=3 7)     G            2 160-28 560       1 388           2-21
Fenêtres double VIR        G            2 550-16 050         196          13-82
PAC air-eau                F            9 000-18 000       3 125            3-6
Chauffe-eau thermodyn.     G             2 500-4 500          85          29-53
VMC hygro B                G             1 500-4 000         -24           > 30


## Bouquets prédéfinis

⚠️ Point pédagogique : les économies d'un bouquet **ne sont pas la somme** des économies
des gestes pris isolément (interactions entre enveloppe, apports gratuits et intermittence).

In [3]:
for nom, mesures in predefined_bundles().items():
    a = assess_scenario(house, list(mesures))
    cout = f"{round(a.investment_low_eur):,}-{round(a.investment_high_eur):,} €".replace(",", " ")
    retour = (f"{a.payback_years_low:.0f} à {a.payback_years_high:.0f} ans"
              if a.payback_years_high != float("inf") else "> 30 ans")
    savings_pct = 100 * (1 - a.after.final_energy_kwh_m2 / a.before.final_energy_kwh_m2)
    print(f"— {nom}")
    print(f"   {a.before.label} → {a.after.label} | conso finale -{savings_pct:.0f} % | "
          f"{round(a.annual_savings_eur)} €/an économisés | "
          f"ΔCO₂ -{round(a.annual_co2_savings_kg)} kg/an")
    print(f"   investissement {cout} | retour simple {retour}")
    print()

— enveloppe_d_abord
   G → F | conso finale -54 % | 3133 €/an économisés | ΔCO₂ -11105 kg/an
   investissement 6 410-53 910 € | retour simple 2 à 17 ans

— sortie_du_fioul
   G → F | conso finale -74 % | 3157 €/an économisés | ΔCO₂ -19247 kg/an
   investissement 11 500-22 500 € | retour simple 4 à 7 ans

— renovation_globale
   G → C | conso finale -88 % | 4568 €/an économisés | ΔCO₂ -19937 kg/an
   investissement 19 410-80 410 € | retour simple 4 à 18 ans

